# Fine-tuning Qwen3.5-4B-Base untuk Domain Hukum Indonesia

Notebook ini menjalankan dua tahap training lokal dengan QLoRA:

1. **DAPT**: corpus regulasi Indonesia menggunakan kolom `text`.
2. **SFT**: QA hukum menggunakan `prompt` → user dan `completion` → assistant.

Dataset sumber dibaca dari snapshot lokal dan tidak diubah. Mode default adalah `smoke`; gunakan `QWEN_LEGAL_RUN_MODE=full` untuk full run. Checkpoint menyimpan adapter dan state training agar dapat dilanjutkan setelah interupsi.

In [ ]:
%env QWEN_LEGAL_RUN_MODE=full

In [ ]:
import gc
import importlib.metadata as importlib_metadata
import math
import json
import os
import platform
import random
import re
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch

SEED = 42
RUN_MODE = os.environ.get('QWEN_LEGAL_RUN_MODE', 'smoke').strip().lower()
if RUN_MODE not in {'smoke', 'full'}:
    raise ValueError(f'QWEN_LEGAL_RUN_MODE harus smoke atau full, bukan {RUN_MODE!r}')

# Smoke default mengikuti rencana checkpoint 250 langkah; override ini berguna untuk
# preflight cepat, misalnya QWEN_LEGAL_SMOKE_STEPS=1.
if RUN_MODE == 'smoke':
    MAX_STEPS = int(os.environ.get('QWEN_LEGAL_SMOKE_STEPS', '250'))
    TRAIN_EXAMPLES = MAX_STEPS * 8
    SAVE_STEPS = MAX_STEPS if MAX_STEPS < 250 else 250
    EVAL_EXAMPLES = 128
else:
    MAX_STEPS = 10_000
    TRAIN_EXAMPLES = 80_000
    SAVE_STEPS = 500
    EVAL_EXAMPLES = 1_024

MAX_LENGTH = 2_048
PER_DEVICE_BATCH = 1
GRADIENT_ACCUMULATION = 8
EVAL_STEPS = SAVE_STEPS
SAVE_TOTAL_LIMIT = 3
LOGGING_STEPS = 10

HF_CACHE = Path.home() / '.cache' / 'huggingface' / 'hub'
MODEL_REVISION = '1001bb4d826a52d1f399e183466143f4da7b741b'
CORPUS_REVISION = '814f32015b10bf376907aa26ce1c12fe8bef700b'
SFT_REVISION = '0d25efe8bf09dad69c3544d9bf62036967508bda'
MODEL_SNAPSHOT = HF_CACHE / 'models--Qwen--Qwen3.5-4B-Base' / 'snapshots' / MODEL_REVISION
CORPUS_SNAPSHOT = HF_CACHE / 'datasets--morpknight--indonesian-legal-corpus' / 'snapshots' / CORPUS_REVISION
SFT_SNAPSHOT = HF_CACHE / 'datasets--morpknight--indonesian-legal-qa-sft' / 'snapshots' / SFT_REVISION

RUN_ROOT = Path(os.environ.get('QWEN_LEGAL_RUN_ROOT', str(Path.cwd() / 'artifacts' / 'local' / 'qwen35_legal_runs')))
RUN_DIR = RUN_ROOT / RUN_MODE
RUN_DIR.mkdir(parents=True, exist_ok=True)

CORPUS_REQUIRED_COLUMNS = {
    'id', 'text', 'regulation_key', 'regulation_type', 'enacting_body',
    'regulation_number', 'year', 'title', 'chapter', 'article', 'domain',
    'chunk_index', 'chunk_count', 'token_count', 'content_hash',
    'source_dataset', 'source_revision', 'source_row_id',
}
SFT_REQUIRED_COLUMNS = {
    'id', 'prompt', 'completion', 'regulation_key', 'answer_hash',
    'question_variant_rank', 'token_count', 'source_dataset',
    'source_revision', 'source_row_id',
}
EXPECTED_COUNTS = {
    'corpus': {'train': 587_070, 'validation': 30_983, 'test': 23_932},
    'qa': {'train': 10_576_428, 'validation': 626_393, 'test': 638_610},
}

LORA_TARGETS = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'in_proj_qkv', 'in_proj_z', 'out_proj',
    'gate_proj', 'up_proj', 'down_proj',
]
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

DAPT_LR = 1e-4
SFT_LR = 2e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print({
    'run_mode': RUN_MODE,
    'max_steps_per_stage': MAX_STEPS,
    'train_examples_per_stage': TRAIN_EXAMPLES,
    'save_steps': SAVE_STEPS,
    'run_dir': str(RUN_DIR),
})

In [ ]:
from datasets import Dataset, load_dataset
import pyarrow as pa
import pyarrow.parquet as pq

def package_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return 'not-installed'

if not torch.cuda.is_available():
    raise RuntimeError('CUDA tidak tersedia; notebook ini membutuhkan GPU lokal.')

gpu = torch.cuda.get_device_properties(0)
if not torch.cuda.is_bf16_supported():
    raise RuntimeError('GPU tidak melaporkan dukungan BF16 yang diperlukan.')

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('Torch:', torch.__version__, 'CUDA runtime:', torch.version.cuda)
print('GPU:', gpu.name, 'capability:', torch.cuda.get_device_capability(0))
print('VRAM GiB:', round(gpu.total_memory / 2**30, 2), 'BF16:', torch.cuda.is_bf16_supported())
print('Transformers:', package_version('transformers'))
print('Datasets:', package_version('datasets'))
print('TRL:', package_version('trl'))
print('PEFT:', package_version('peft'))
print('BitsAndBytes:', package_version('bitsandbytes'))

def assert_snapshot(path, label):
    if not path.is_dir():
        raise FileNotFoundError(f'Snapshot {label} tidak ditemukan: {path}')
    print(f'{label}: {path}')

assert_snapshot(MODEL_SNAPSHOT, 'model')
assert_snapshot(CORPUS_SNAPSHOT, 'corpus')
assert_snapshot(SFT_SNAPSHOT, 'qa-sft')

def parquet_files(snapshot, split):
    candidates = sorted((snapshot / 'data').glob(f'{split}-*.parquet'))
    if not candidates:
        candidates = sorted(snapshot.glob(f'**/{split}-*.parquet'))
    if not candidates:
        raise FileNotFoundError(f'Tidak ada parquet untuk split {split} di {snapshot}')
    return candidates

def parquet_row_count(paths):
    return sum(pq.ParquetFile(path).metadata.num_rows for path in paths)

def load_local_split(snapshot, split, training_columns):
    paths = parquet_files(snapshot, split)
    full_row_count = parquet_row_count(paths)
    selected_paths = paths if RUN_MODE == 'full' else paths[:1]
    mode_label = 'full' if RUN_MODE == 'full' else 'smoke-first-shard'
    print(f'Memuat {split}: {len(selected_paths)}/{len(paths)} shard ({mode_label}); full rows={full_row_count:,}')
    if RUN_MODE == 'smoke':
        target_rows = TRAIN_EXAMPLES if split == 'train' else EVAL_EXAMPLES if split == 'validation' else 1
        batch = next(pq.ParquetFile(selected_paths[0]).iter_batches(batch_size=target_rows, columns=list(training_columns)))
        return Dataset(pa.Table.from_batches([batch]))
    return load_dataset(
        'parquet',
        data_files={split: [str(path) for path in selected_paths]},
        split=split,
        keep_in_memory=False,
    )

corpus = {split: load_local_split(CORPUS_SNAPSHOT, split, ['text']) for split in ('train', 'validation', 'test')}
qa = {split: load_local_split(SFT_SNAPSHOT, split, ['prompt', 'completion']) for split in ('train', 'validation', 'test')}

PARQUET_COUNTS = {
    'corpus': {split: parquet_row_count(parquet_files(CORPUS_SNAPSHOT, split)) for split in ('train', 'validation', 'test')},
    'qa': {split: parquet_row_count(parquet_files(SFT_SNAPSHOT, split)) for split in ('train', 'validation', 'test')},
}
for name, counts in PARQUET_COUNTS.items():
    for split, count in counts.items():
        if count != EXPECTED_COUNTS[name][split]:
            raise ValueError(f'Parquet metadata {name}/{split}={count}, expected {EXPECTED_COUNTS[name][split]}')

PARQUET_COLUMNS = {
    'corpus': {split: set(pq.ParquetFile(parquet_files(CORPUS_SNAPSHOT, split)[0]).schema_arrow.names) for split in ('train', 'validation', 'test')},
    'qa': {split: set(pq.ParquetFile(parquet_files(SFT_SNAPSHOT, split)[0]).schema_arrow.names) for split in ('train', 'validation', 'test')},
}
for name, columns_by_split in PARQUET_COLUMNS.items():
    required = CORPUS_REQUIRED_COLUMNS if name == 'corpus' else SFT_REQUIRED_COLUMNS
    for split, columns in columns_by_split.items():
        missing = required - columns
        if missing:
            raise ValueError(f'Schema parquet {name}/{split} kehilangan kolom: {sorted(missing)}')

def validate_dataset(name, dataset_dict, required_columns):
    for split, dataset in dataset_dict.items():
        actual = set(dataset.column_names)
        expected_input_columns = {'text'} if name == 'corpus' else {'prompt', 'completion'}
        missing = (required_columns if RUN_MODE == 'full' else expected_input_columns) - actual
        if missing:
            raise ValueError(f'{name}/{split} kehilangan kolom: {sorted(missing)}')
        expected = EXPECTED_COUNTS[name][split]
        if RUN_MODE == 'full' and len(dataset) != expected:
            raise ValueError(f'{name}/{split} memiliki {len(dataset)} baris, expected {expected}')
        minimum_needed = TRAIN_EXAMPLES if split == 'train' else EVAL_EXAMPLES if split == 'validation' else 1
        if len(dataset) < minimum_needed:
            raise ValueError(f'{name}/{split} shard smoke hanya memiliki {len(dataset)} baris; perlu minimal {minimum_needed}')
        print(f'{name}/{split}: loaded={len(dataset):,}; full_expected={expected:,}; columns={len(actual)}')

validate_dataset('corpus', corpus, CORPUS_REQUIRED_COLUMNS)
validate_dataset('qa', qa, SFT_REQUIRED_COLUMNS)


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_SNAPSHOT,
    local_files_only=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def deterministic_subset(dataset, size, seed):
    size = min(int(size), len(dataset))
    if size <= 0:
        raise ValueError('Ukuran subset harus lebih besar dari nol.')
    rng = random.Random(seed)
    indices = sorted(rng.sample(range(len(dataset)), size))
    return dataset.select(indices).shuffle(seed=seed)

corpus_train = deterministic_subset(corpus['train'], TRAIN_EXAMPLES, SEED)
corpus_eval = deterministic_subset(corpus['validation'], EVAL_EXAMPLES, SEED + 1)
qa_train_raw = deterministic_subset(qa['train'], TRAIN_EXAMPLES, SEED + 2)
qa_eval_raw = deterministic_subset(qa['validation'], EVAL_EXAMPLES, SEED + 3)

def qa_to_messages(example):
    return {
        'messages': [
            {'role': 'user', 'content': str(example['prompt'])},
            {'role': 'assistant', 'content': str(example['completion'])},
        ]
    }

qa_train = qa_train_raw.map(
    qa_to_messages,
    remove_columns=qa_train_raw.column_names,
    desc='Mapping prompt/completion menjadi messages untuk SFT',
)
qa_eval = qa_eval_raw.map(
    qa_to_messages,
    remove_columns=qa_eval_raw.column_names,
    desc='Mapping validation QA menjadi messages',
)

assert 'text' in corpus_train.column_names
assert qa_train.column_names == ['messages']
assert qa_eval.column_names == ['messages']
print('DAPT mapping: kolom text digunakan langsung; metadata tidak dimasukkan ke input model.')
print('SFT mapping:', qa_train[0]['messages'])
rendered = tokenizer.apply_chat_template(qa_train[0]['messages'], tokenize=True, add_generation_prompt=False, return_dict=True)
rendered_ids = rendered['input_ids']
rendered_token_count = int(rendered_ids.shape[-1]) if hasattr(rendered_ids, 'shape') else len(rendered_ids)
print('SFT rendered token count:', rendered_token_count)

del qa_train_raw, qa_eval_raw
gc.collect()

In [ ]:
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

def make_quantization_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

def load_base_model():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_SNAPSHOT,
        quantization_config=make_quantization_config(),
        dtype=torch.bfloat16,
        device_map={'': 'cuda:0'},
        low_cpu_mem_usage=True,
        local_files_only=True,
    )
    model.config.use_cache = False
    model.config.pad_token_id = tokenizer.pad_token_id
    if getattr(model, 'generation_config', None) is not None:
        model.generation_config.pad_token_id = tokenizer.pad_token_id
    return model

def make_lora_config():
    return LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias='none',
        task_type='CAUSAL_LM',
        target_modules=LORA_TARGETS,
    )

def assert_lora_targets_exist(model):
    module_suffixes = {name.rsplit('.', 1)[-1] for name, _ in model.named_modules()}
    missing = sorted(set(LORA_TARGETS) - module_suffixes)
    if missing:
        raise ValueError(f'Target LoRA tidak ditemukan pada model: {missing}')

def trainer_config(stage, output_dir, max_steps=None, save_steps=None, eval_steps=None):
    is_sft = stage == 'sft'
    return SFTConfig(
        output_dir=str(output_dir),
        max_steps=MAX_STEPS if max_steps is None else int(max_steps),
        per_device_train_batch_size=PER_DEVICE_BATCH,
        per_device_eval_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        learning_rate=SFT_LR if is_sft else DAPT_LR,
        lr_scheduler_type='cosine',
        warmup_steps=max(1, int(round((MAX_STEPS if max_steps is None else int(max_steps)) * 0.03))),
        weight_decay=0.01,
        max_grad_norm=1.0,
        optim='adamw_torch_fused',
        logging_strategy='steps',
        logging_steps=LOGGING_STEPS,
        logging_first_step=True,
        eval_strategy='steps',
        eval_steps=EVAL_STEPS if eval_steps is None else int(eval_steps),
        save_strategy='steps',
        save_steps=SAVE_STEPS if save_steps is None else int(save_steps),
        save_total_limit=SAVE_TOTAL_LIMIT,
        save_only_model=False,
        bf16=True,
        tf32=True,
        gradient_checkpointing=True,
        max_length=MAX_LENGTH,
        packing=False,
        dataset_text_field='text',
        assistant_only_loss=is_sft,
        completion_only_loss=False,
        remove_unused_columns=False,
        report_to='none',
        seed=SEED,
        data_seed=SEED,
        dataloader_num_workers=0,
    )

print('LoRA targets akan divalidasi saat setiap stage membuat model:', LORA_TARGETS)

In [ ]:
CHECKPOINT_PATTERN = re.compile(r'^checkpoint-(\d+)$')

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + '.tmp')
    temp_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    temp_path.replace(path)

def environment_manifest():
    info = {
        'python': sys.version,
        'platform': platform.platform(),
        'torch': torch.__version__,
        'cuda_runtime': torch.version.cuda,
        'transformers': package_version('transformers'),
        'datasets': package_version('datasets'),
        'trl': package_version('trl'),
        'peft': package_version('peft'),
        'bitsandbytes': package_version('bitsandbytes'),
        'gpu': torch.cuda.get_device_name(0),
        'gpu_capability': list(torch.cuda.get_device_capability(0)),
        'gpu_memory_bytes': torch.cuda.get_device_properties(0).total_memory,
        'bf16_supported': torch.cuda.is_bf16_supported(),
    }
    return info

manifest = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'run_mode': RUN_MODE,
    'run_dir': str(RUN_DIR),
    'model': {
        'name': 'Qwen/Qwen3.5-4B-Base',
        'snapshot': str(MODEL_SNAPSHOT),
        'revision': MODEL_REVISION,
    },
    'datasets': {
        'corpus': {'name': 'morpknight/indonesian-legal-corpus', 'snapshot': str(CORPUS_SNAPSHOT), 'revision': CORPUS_REVISION},
        'qa_sft': {'name': 'morpknight/indonesian-legal-qa-sft', 'snapshot': str(SFT_SNAPSHOT), 'revision': SFT_REVISION},
    },
    'mapping': {
        'dapt': {'input': ['text'], 'objective': 'causal_lm_all_tokens'},
        'sft': {'input': ['prompt', 'completion'], 'output': 'messages[user, assistant]', 'objective': 'assistant_only_loss'},
    },
    'hyperparameters': {
        'max_steps_per_stage': MAX_STEPS,
        'train_examples_per_stage': TRAIN_EXAMPLES,
        'max_length': MAX_LENGTH,
        'per_device_train_batch_size': PER_DEVICE_BATCH,
        'gradient_accumulation_steps': GRADIENT_ACCUMULATION,
        'save_steps': SAVE_STEPS,
        'save_total_limit': SAVE_TOTAL_LIMIT,
        'dapt_learning_rate': DAPT_LR,
        'sft_learning_rate': SFT_LR,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'lora_dropout': LORA_DROPOUT,
        'lora_target_modules': LORA_TARGETS,
        'packing': False,
        'bf16': True,
        'quantization': '4bit nf4 double-quant',
    },
    'environment': environment_manifest(),
    'source_row_counts': PARQUET_COUNTS,
    'loaded_row_counts': {
        'corpus': {split: len(dataset) for split, dataset in corpus.items()},
        'qa': {split: len(dataset) for split, dataset in qa.items()},
    },
    'stages': {},
}
write_json(RUN_DIR / 'run_manifest.json', manifest)
print('Manifest awal ditulis:', RUN_DIR / 'run_manifest.json')

def checkpoint_dirs(stage):
    stage_dir = RUN_DIR / stage
    if not stage_dir.exists():
        return []
    found = []
    for path in stage_dir.iterdir():
        match = CHECKPOINT_PATTERN.match(path.name)
        if match and path.is_dir() and (path / 'trainer_state.json').exists() and (path / 'adapter_config.json').exists():
            found.append((int(match.group(1)), path))
    return [path for _, path in sorted(found)]

def latest_checkpoint(stage):
    paths = checkpoint_dirs(stage)
    return paths[-1] if paths else None

def checkpoint_step(path):
    state_path = Path(path) / 'trainer_state.json'
    if not state_path.exists():
        return None
    return int(json.loads(state_path.read_text(encoding='utf-8')).get('global_step', 0))

def validate_checkpoint_adapter(path):
    adapter_path = Path(path) / 'adapter_config.json'
    if not adapter_path.exists():
        raise FileNotFoundError(f'adapter_config.json tidak ada di {path}')
    actual = json.loads(adapter_path.read_text(encoding='utf-8'))
    expected_targets = sorted(LORA_TARGETS)
    actual_targets = sorted(list(actual.get('target_modules', [])))
    checks = {
        'r': (actual.get('r'), LORA_R),
        'lora_alpha': (actual.get('lora_alpha'), LORA_ALPHA),
        'lora_dropout': (actual.get('lora_dropout'), LORA_DROPOUT),
        'target_modules': (actual_targets, expected_targets),
        'bias': (actual.get('bias'), 'none'),
    }
    mismatches = {key: values for key, values in checks.items() if values[0] != values[1]}
    if mismatches:
        raise ValueError(f'Konfigurasi LoRA checkpoint tidak kompatibel: {mismatches}')
    return actual

def update_manifest_stage(stage, payload):
    manifest['stages'][stage] = payload
    manifest['updated_at_utc'] = datetime.now(timezone.utc).isoformat()
    write_json(RUN_DIR / 'run_manifest.json', manifest)


In [ ]:
def build_dapt_trainer(model, output_dir):
    return SFTTrainer(
        model=model,
        args=trainer_config('dapt', output_dir),
        train_dataset=corpus_train,
        eval_dataset=corpus_eval,
        processing_class=tokenizer,
        peft_config=make_lora_config(),
    )

def build_sft_trainer(model, output_dir, use_new_adapter):
    return SFTTrainer(
        model=model,
        args=trainer_config('sft', output_dir),
        train_dataset=qa_train,
        eval_dataset=qa_eval,
        processing_class=tokenizer,
        peft_config=make_lora_config() if use_new_adapter else None,
    )

def save_final_adapter(trainer_or_model, stage):
    final_dir = RUN_DIR / stage / 'final_adapter'
    final_dir.mkdir(parents=True, exist_ok=True)
    if hasattr(trainer_or_model, 'save_model'):
        trainer_or_model.save_model(str(final_dir))
    else:
        trainer_or_model.save_pretrained(str(final_dir))
    tokenizer.save_pretrained(str(final_dir))
    return final_dir

def load_adapter_for_inference(adapter_dir):
    model = load_base_model()
    model = PeftModel.from_pretrained(model, str(adapter_dir), is_trainable=False)
    model.config.use_cache = True
    model.eval()
    return model


## Stage 1 — DAPT

Training menggunakan `legal_corpus['text']` dan causal language-model loss pada seluruh token. Jika notebook dijalankan kembali, checkpoint terakhir yang valid akan digunakan otomatis.

In [ ]:
dapt_dir = RUN_DIR / 'dapt'
dapt_dir.mkdir(parents=True, exist_ok=True)
dapt_checkpoint = latest_checkpoint('dapt')
if dapt_checkpoint is not None:
    validate_checkpoint_adapter(dapt_checkpoint)
    print('DAPT checkpoint ditemukan:', dapt_checkpoint, 'step:', checkpoint_step(dapt_checkpoint))

dapt_final = dapt_dir / 'final_adapter'
dapt_step = checkpoint_step(dapt_checkpoint) if dapt_checkpoint else 0
if dapt_checkpoint is not None and dapt_step >= MAX_STEPS and dapt_final.exists():
    print('DAPT sudah selesai; menggunakan final adapter:', dapt_final)
    update_manifest_stage('dapt', {
        'status': 'already_complete',
        'final_adapter': str(dapt_final),
        'last_checkpoint': str(dapt_checkpoint),
        'global_step': dapt_step,
    })
elif dapt_checkpoint is not None and dapt_step >= MAX_STEPS:
    dapt_model = load_base_model()
    dapt_model = PeftModel.from_pretrained(dapt_model, str(dapt_checkpoint), is_trainable=False)
    dapt_final = save_final_adapter(dapt_model, 'dapt')
    print('DAPT final dipulihkan dari checkpoint:', dapt_final)
    update_manifest_stage('dapt', {
        'status': 'restored_from_checkpoint',
        'final_adapter': str(dapt_final),
        'last_checkpoint': str(dapt_checkpoint),
        'global_step': dapt_step,
    })
    del dapt_model
    gc.collect()
    torch.cuda.empty_cache()
else:
    dapt_model = load_base_model()
    assert_lora_targets_exist(dapt_model)
    dapt_trainer = build_dapt_trainer(dapt_model, dapt_dir)
    if dapt_checkpoint is not None and dapt_step < MAX_STEPS:
        dapt_result = dapt_trainer.train(resume_from_checkpoint=str(dapt_checkpoint))
    elif dapt_checkpoint is None:
        dapt_result = dapt_trainer.train()
    else:
        dapt_result = None
    dapt_final = save_final_adapter(dapt_trainer, 'dapt')
    dapt_metrics = dict(dapt_result.metrics) if dapt_result is not None else {}
    print('DAPT selesai:', dapt_final)
    print('DAPT metrics:', dapt_metrics)
    update_manifest_stage('dapt', {
        'status': 'completed',
        'final_adapter': str(dapt_final),
        'last_checkpoint': str(latest_checkpoint('dapt')) if latest_checkpoint('dapt') else None,
        'global_step': checkpoint_step(latest_checkpoint('dapt')) if latest_checkpoint('dapt') else MAX_STEPS,
        'metrics': dapt_metrics,
    })
    del dapt_trainer, dapt_model
    gc.collect()
    torch.cuda.empty_cache()

## Stage 2 — SFT

Training melanjutkan adapter hasil DAPT. Dataset QA dipakai sebagai conversational `messages`, dengan loss hanya pada token assistant.

In [ ]:
sft_dir = RUN_DIR / 'sft'
sft_dir.mkdir(parents=True, exist_ok=True)
sft_checkpoint = latest_checkpoint('sft')
if sft_checkpoint is not None:
    validate_checkpoint_adapter(sft_checkpoint)
    print('SFT checkpoint ditemukan:', sft_checkpoint, 'step:', checkpoint_step(sft_checkpoint))

sft_final = sft_dir / 'final_adapter'
sft_step = checkpoint_step(sft_checkpoint) if sft_checkpoint else 0
sft_result = None

if sft_checkpoint is not None and sft_step >= MAX_STEPS and sft_final.exists():
    print('SFT sudah selesai; menggunakan final adapter:', sft_final)
    update_manifest_stage('sft', {
        'status': 'already_complete',
        'final_adapter': str(sft_final),
        'last_checkpoint': str(sft_checkpoint),
        'global_step': sft_step,
    })
elif sft_checkpoint is not None and sft_step >= MAX_STEPS:
    sft_model = load_base_model()
    sft_model = PeftModel.from_pretrained(sft_model, str(sft_checkpoint), is_trainable=False)
    sft_final = save_final_adapter(sft_model, 'sft')
    del sft_model
    gc.collect()
    torch.cuda.empty_cache()
else:
    if sft_checkpoint is not None:
        sft_model = load_base_model()
        sft_trainer = build_sft_trainer(sft_model, sft_dir, use_new_adapter=True)
        sft_result = sft_trainer.train(resume_from_checkpoint=str(sft_checkpoint))
    else:
        if not dapt_final.exists():
            raise FileNotFoundError(f'Adapter DAPT tidak ditemukan: {dapt_final}')
        validate_checkpoint_adapter(dapt_final) if (dapt_final / 'adapter_config.json').exists() else None
        sft_model = load_base_model()
        sft_model = PeftModel.from_pretrained(sft_model, str(dapt_final), is_trainable=True)
        sft_trainer = build_sft_trainer(sft_model, sft_dir, use_new_adapter=False)
        sft_result = sft_trainer.train()
    sft_final = save_final_adapter(sft_trainer, 'sft')
    sft_metrics = dict(sft_result.metrics) if sft_result is not None else {}
    print('SFT selesai:', sft_final)
    print('SFT metrics:', sft_metrics)
    update_manifest_stage('sft', {
        'status': 'completed',
        'final_adapter': str(sft_final),
        'last_checkpoint': str(latest_checkpoint('sft')) if latest_checkpoint('sft') else None,
        'global_step': checkpoint_step(latest_checkpoint('sft')) if latest_checkpoint('sft') else MAX_STEPS,
        'metrics': sft_metrics,
    })
    del sft_trainer, sft_model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# Smoke-only acceptance checks. Full run tidak mengulang satu step tambahan.
if RUN_MODE == 'smoke':
    final_checkpoint = latest_checkpoint('sft')
    if final_checkpoint is None:
        raise RuntimeError('Smoke SFT tidak menghasilkan checkpoint.')
    validate_checkpoint_adapter(final_checkpoint)
    if checkpoint_step(final_checkpoint) < MAX_STEPS:
        raise RuntimeError(f'Checkpoint SFT berhenti di step {checkpoint_step(final_checkpoint)}')
    required_files = ['adapter_config.json', 'adapter_model.safetensors', 'trainer_state.json']
    missing = [name for name in required_files if not (final_checkpoint / name).exists()]
    if missing:
        raise FileNotFoundError(f'File checkpoint smoke hilang: {missing}')

    # Resume probe memakai konfigurasi LoRA yang sama dan output terpisah.
    probe_dir = RUN_DIR / 'resume_probe'
    probe_dir.mkdir(parents=True, exist_ok=True)
    probe_model = load_base_model()
    probe_trainer = SFTTrainer(
        model=probe_model,
        args=trainer_config('sft', probe_dir, max_steps=MAX_STEPS + 1, save_steps=SAVE_STEPS, eval_steps=EVAL_STEPS),
        train_dataset=qa_train,
        eval_dataset=qa_eval,
        processing_class=tokenizer,
        peft_config=make_lora_config(),
    )
    probe_result = probe_trainer.train(resume_from_checkpoint=str(final_checkpoint))
    if probe_trainer.state.global_step != MAX_STEPS + 1:
        raise RuntimeError(f'Resume probe tidak melanjutkan step: {probe_trainer.state.global_step}')
    print('Resume probe berhasil; global_step:', probe_trainer.state.global_step)
    del probe_trainer, probe_model
    gc.collect()
    torch.cuda.empty_cache()

# Generation smoke dengan adapter final.
inference_model = load_adapter_for_inference(sft_final)
sample_messages = [
    {'role': 'user', 'content': 'Apa fungsi utama suatu peraturan perundang-undangan dalam sistem hukum Indonesia?'},
]
encoded = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
    return_dict=True,
)
encoded = {key: value.to('cuda:0') for key, value in encoded.items()}
prompt_length = encoded['input_ids'].shape[-1]
with torch.inference_mode():
    generated = inference_model.generate(
        **encoded,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
answer = tokenizer.decode(generated[0][prompt_length:], skip_special_tokens=True)
print('Generation smoke output:\n', answer[:1000])

update_manifest_stage('acceptance', {
    'status': 'passed',
    'latest_sft_checkpoint': str(latest_checkpoint('sft')) if latest_checkpoint('sft') else None,
    'final_adapter': str(sft_final),
    'generation_chars': len(answer),
})
print('Semua acceptance checks selesai. Manifest:', RUN_DIR / 'run_manifest.json')